# Bài 01 · Perceptron: một nơ-ron đọc chữ số viết tay

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/01-perceptron.ipynb)

Notebook đi kèm [Bài 01 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/perceptron/) (reel và demo tương tác nằm ở trang bài học).
Ta tự viết bằng numpy, không dùng thư viện học máy, một nơ-ron học trả lời câu hỏi
**“ảnh này là số 7 hay số 1?”** trên dữ liệu MNIST thật.

**Bạn sẽ làm:**
1. Biến một ảnh 28×28 thành vectơ 784 đầu vào $x$.
2. Tính $z = \sum_i x_i w_i + b$ (784 phép nhân–cộng) và quyết định bằng hàm bậc thang.
3. Huấn luyện bằng quy tắc của Rosenblatt: chỉ sửa trọng số khi đoán sai.
4. Đọc trọng số đã học như một bức ảnh, xem mô hình sai ở đâu.
5. Thấy giới hạn của một nơ-ron (XOR) và bước từ một nơ-ron lên một lớp $z = xW + b$.

**Cần biết trước:** Python và numpy cơ bản (mảng, `@`, chỉ mục). **Thời gian chạy:** dưới 1 phút trên CPU,
kể cả tải MNIST (~11 MB) lần đầu.

Ký hiệu theo chương *Neural Computation* của [MLSysBook](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html).
Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong reel và demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn (13.007; 99,35%). Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import itertools
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm hình dạng khác nhau: "1" = tròn, "7" = vuông
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527))

## 2 · Dữ liệu MNIST

MNIST (LeCun và cộng sự, 1998) gồm 70.000 ảnh chữ số viết tay xám 28×28: 60.000 ảnh huấn luyện, 10.000 ảnh kiểm tra.
Ô dưới tải 4 tệp gốc từ một bản sao công khai và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
print("huấn luyện:", train_img.shape, " kiểm tra:", test_img.shape, " kiểu:", train_img.dtype)

Hình dưới là 20 ảnh huấn luyện đầu tiên, mỗi ảnh ghi nhãn đúng của nó ở trên.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(10, 2.4))
for ax, img, lab in zip(axes.flat, train_img, train_lab):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")
plt.show()

## 3 · Từ ảnh đến vectơ $x$

Một nơ-ron không “nhìn” ảnh; nó nhận một dãy số. Ta trải 28 hàng của ảnh nối đuôi nhau thành
vectơ $x = [x_1, \dots, x_{784}]$, và chia cho 255 để mỗi $x_i$ nằm trong $[0, 1]$: 0 là nền trắng, 1 là nét đậm nhất.

Ta chỉ giữ ảnh số 7 và số 1, và đặt nhãn $y = 1$ cho “7”, $y = 0$ cho “1”.

In [ ]:
POS, NEG = 7, 1  # y = 1 nghĩa là "7", y = 0 nghĩa là "1"


def binary_set(images, labels, pos=POS, neg=NEG):
    keep = (labels == pos) | (labels == neg)
    X = images[keep].reshape(-1, 784) / 255.0  # mỗi hàng là một ảnh: 784 số trong [0, 1]
    y = (labels[keep] == pos).astype(int)
    return X, y, np.nonzero(keep)[0]  # chỉ số gốc trong MNIST, để tra lại ảnh


Xtr, ytr, idx_tr = binary_set(train_img, train_lab)
Xte, yte, idx_te = binary_set(test_img, test_lab)
print(
    f"huấn luyện: {vn(len(ytr), 0)} ảnh ({vn(ytr.sum(), 0)} số 7)  ·  kiểm tra: {vn(len(yte), 0)} ảnh ({vn(yte.sum(), 0)} số 7)"
)
print("một ảnh là một vectơ dài", Xtr[0].shape[0])
assert (len(ytr), ytr.sum(), len(yte), yte.sum()) == (13007, 6265, 2163, 1028)

Thử nhìn một ảnh ở cả hai dạng: lưới 28×28 và dải 784 số (hàng 1 của ảnh là 28 số đầu, hàng 2 là 28 số kế tiếp…).
Phần lớn $x_i$ bằng 0 vì là nền; chỉ những điểm có nét mới khác 0.

In [ ]:
x = Xtr[0]
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 2.6), gridspec_kw={"width_ratios": [1, 4]})
a.imshow(x.reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
a.set_title(f"ảnh 28×28 (số {POS if ytr[0] else NEG})")
a.axis("off")
b.plot(np.arange(1, 785), x, color=BLUE, lw=1)
b.set(xlabel="chỉ số i (1…784)", ylabel="xᵢ", xlim=(1, 784), title="cùng ảnh đó, trải thành vectơ x")
plt.show()
print(f"{(x > 0).sum()} trên 784 điểm ảnh có nét")

## 4 · Một nơ-ron: 784 phép nhân–cộng

Perceptron gán cho mỗi đầu vào $x_i$ một trọng số $w_i$, cộng tất cả cùng độ lệch $b$, rồi quyết định:

$$z = \sum_{i=1}^{784} x_i\, w_i + b, \qquad \hat y = f(z) = \begin{cases} 1 \;(\text{“7”}) & z \ge 0 \\ 0 \;(\text{“1”}) & z < 0 \end{cases}$$

Mỗi số hạng $x_i w_i$ là một **phép nhân–cộng (MAC)**: nhân, rồi cộng dồn vào tổng.
Viết bằng vòng lặp cho thấy rõ điều đó; numpy làm cùng việc bằng một tích vô hướng `x @ w`.

In [ ]:
rng = np.random.default_rng(42)
w_demo = rng.normal(0, 0.1, 784)  # trọng số ngẫu nhiên, chỉ để minh họa phép tính
b_demo = 0.5


def neuron_loop(x, w, b):
    z, macs = b, 0
    for xi, wi in zip(x, w):
        z += xi * wi  # một phép nhân–cộng
        macs += 1
    return z, macs


def step(z):
    return (z >= 0).astype(int)


z_loop, macs = neuron_loop(x, w_demo, b_demo)
z_vec = x @ w_demo + b_demo
assert np.isclose(z_loop, z_vec)
print(f"z = {vn(z_vec, 4)} sau {macs} MAC  →  ŷ = {step(np.array(z_vec))}")

In [ ]:
t0 = time.perf_counter()
for x_ in Xte[:200]:
    neuron_loop(x_, w_demo, b_demo)
t_loop = (time.perf_counter() - t0) / 200
t0 = time.perf_counter()
Xte @ w_demo + b_demo  # cả tập kiểm tra trong một lần
t_vec = (time.perf_counter() - t0) / len(Xte)
print(
    f"vòng lặp Python: {vn(t_loop * 1e6, 0)} µs/ảnh  ·  numpy: {vn(t_vec * 1e6, 2)} µs/ảnh  (nhanh hơn ~{vn(t_loop / t_vec, 0)} lần)"
)

Phép tính hoàn toàn giống nhau; khác biệt là numpy chạy 784 MAC bằng mã máy được tối ưu, và xử lý cả
2.163 ảnh như một phép nhân ma trận. Đây là lý do mạng nơ-ron được viết bằng phép nhân ma trận (mục 10).

### 🤔 Dự đoán trước
Trước khi học, mọi trọng số đều bằng 0: $w = 0$, $b = 0$. Khi đó nơ-ron trả lời gì cho mỗi ảnh,
và đúng bao nhiêu phần trăm tập kiểm tra?

In [ ]:
w0, b0 = np.zeros(784), 0.0
pred0 = step(Xte @ w0 + b0)
print("các dự đoán khác nhau:", np.unique(pred0), " →  độ chính xác", pct((pred0 == yte).mean()))

Với $w = 0$ thì $z = 0$ cho mọi ảnh, và vì $f(0) = 1$, nơ-ron trả lời “7” cho tất cả. Độ chính xác bằng đúng
tỉ lệ số 7 trong tập kiểm tra: 1.028 / 2.163. Đây là mốc phải vượt qua.

## 5 · Học từ lỗi sai: quy tắc Rosenblatt (1958)

Duyệt từng ảnh huấn luyện theo thứ tự ngẫu nhiên. Nếu đoán đúng, không làm gì. Nếu sai, sửa:

$$w_i \leftarrow w_i + \eta\,(y - \hat y)\,x_i, \qquad b \leftarrow b + \eta\,(y - \hat y)$$

- Ảnh là số 7 ($y = 1$) mà đoán “1” ($\hat y = 0$): $y - \hat y = +1$, **cộng** $\eta x$ vào $w$, nên lần sau $z$ của ảnh này lớn hơn.
- Ảnh là số 1 mà đoán “7”: $y - \hat y = -1$, **trừ** $\eta x$, nên $z$ nhỏ đi.

$\eta$ (tốc độ học) là cỡ bước. Ta dùng $\eta = 0{,}1$ và một lượt (epoch) qua 13.007 ảnh, như reel.

In [ ]:
def train(X, y, eta=0.1, seed=0, epochs=1, X_eval=None, y_eval=None):
    """Quy tắc perceptron. Trả về w, b và nhật ký các lần sửa (cùng độ chính xác kiểm tra sau mỗi lần sửa)."""
    w, b = np.zeros(X.shape[1]), 0.0
    order = np.random.default_rng(seed).permutation(len(y))
    log = {"seen": [], "i": [], "z": [], "acc": [], "epoch_errors": []}

    def evaluate():
        if X_eval is not None:
            log["acc"].append(float((step(X_eval @ w + b) == y_eval).mean()))

    evaluate()
    seen = 0
    for _ in range(epochs):
        errors = 0
        for i in order:
            seen += 1
            z = X[i] @ w + b
            y_hat = int(z >= 0)
            if y_hat != y[i]:
                d = y[i] - y_hat  # +1 hoặc −1
                w += eta * d * X[i]
                b += eta * d
                errors += 1
                log["seen"].append(seen)
                log["i"].append(int(i))
                log["z"].append(float(z))
                evaluate()
        log["epoch_errors"].append(errors)
        if errors == 0:  # một lượt không sai ảnh nào: đã tách được hoàn toàn tập huấn luyện
            break
    return w, b, log


t0 = time.perf_counter()
w, b, log = train(Xtr, ytr, eta=0.1, seed=0, X_eval=Xte, y_eval=yte)
pred = step(Xte @ w + b)
print(f"{len(log['seen'])} lần sửa trong {vn(len(ytr), 0)} ảnh ({vn((time.perf_counter() - t0) * 1000, 0)} ms)")
print(
    f"độ chính xác kiểm tra: {pct(log['acc'][0])} → {pct((pred == yte).mean())}"
    f"  ({vn((pred == yte).sum(), 0)}/{vn(len(yte), 0)} ảnh)  ·  b = {vn(b, 2)}"
)
assert len(log["seen"]) == 146 and (pred == yte).sum() == 2149, "khác với reel và demo: hãy kiểm tra lại dữ liệu"

Chỉ **146 lần sửa** (khoảng 1% số ảnh) là đủ đưa độ chính xác từ 47,53% lên **99,35%**, đúng những con số trong
reel và demo. Phần lớn các lần sửa nằm ở đầu: sau vài trăm ảnh, nơ-ron đã đoán đúng gần hết.

Đồ thị dưới vẽ độ chính xác kiểm tra sau mỗi lần sửa (trục ngang là thứ tự lần sửa, không phải số ảnh đã xem).
Đường gạch là mốc 47,53% của $w = 0$.

In [ ]:
acc = np.array(log["acc"])
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(np.arange(len(acc)), 100 * acc, color=BLUE, lw=2)
ax.axhline(100 * acc[0], color=GREY, lw=1, ls="--")
ax.set(xlabel="số lần sửa", ylabel="đúng trên tập kiểm tra (%)", ylim=(40, 101), xlim=(0, len(acc) - 1))
ax.set_title("Độ chính xác sau mỗi lần sửa sai (η = 0,1, seed 0)")
ax.grid(axis="y", alpha=0.3)
plt.show()
print("10 lần sửa đầu:", " · ".join(pct(a, 1) for a in acc[:11]))
seen = np.array(log["seen"])
print(
    f"ảnh thứ mấy gây ra lần sửa: 10 lần đầu nằm trong {seen[9]} ảnh đầu; {(seen <= 1000).sum()} lần trong 1.000 ảnh đầu; "
    f"lần cuối ở ảnh {vn(seen[-1], 0)}"
)

Đường cong không đi lên đều: một lần sửa giúp một ảnh lại có thể làm hỏng nhiều ảnh khác
(lần sửa thứ 4 kéo độ chính xác từ 91,9% xuống 63,4%). Quy tắc chỉ nhìn một ảnh mỗi lần.

### Xem tận mắt 4 lần sửa đầu tiên
Mỗi cột là một lần sửa: ảnh gây lỗi (hàng trên) và $w$ ngay sau khi sửa (hàng dưới; xanh = dương, cam = âm).
Vì $w$ bắt đầu từ 0, $w$ sau lần sửa đầu chính là $-\eta$ lần bức ảnh đó.

In [ ]:
def show_w(ax, w, title=None, vmax=None):
    vmax = vmax or np.abs(w).max() or 1
    im = ax.imshow(w.reshape(28, 28), cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return im


w_step = np.zeros(784)
b_step = 0.0
fig, axes = plt.subplots(2, 4, figsize=(9, 4.6))
for k in range(4):
    i, z = log["i"][k], log["z"][k]
    d = ytr[i] - int(z >= 0)
    w_step += 0.1 * d * Xtr[i]
    b_step += 0.1 * d
    axes[0, k].imshow(Xtr[i].reshape(28, 28), cmap="gray_r")
    axes[0, k].set_title(
        f"ảnh thứ {log['seen'][k]}: số {POS if ytr[i] else NEG}\nz = {vn(z, 2)} → đoán “{POS if z >= 0 else NEG}”",
        fontsize=10,
    )
    axes[0, k].axis("off")
    show_w(axes[1, k], w_step, f"w sau lần sửa {k + 1} ({'+' if d > 0 else '−'}η·x)")
plt.tight_layout()
plt.show()

Lần sửa đầu tiên xảy ra ngay ở ảnh thứ nhất: một số 1 với $z = 0$ bị đoán “7” (vì $f(0) = 1$), nên $w$ trừ đi
$0{,}1$ lần nét số 1 đó. Ảnh thứ 2 là số 1 nữa và $z < 0$, nên đúng và không sửa.

## 6 · Đọc trọng số như một bức ảnh

Mỗi $w_i$ thuộc về một điểm ảnh, nên có thể vẽ $w$ thành ảnh 28×28.
**Xanh** ($w_i > 0$): nét ở đây đẩy $z$ lên, tức nghiêng về “7”. **Cam** ($w_i < 0$): kéo $z$ xuống, nghiêng về “1”.

### 🤔 Dự đoán trước
Bạn nghĩ vùng xanh và vùng cam sẽ nằm ở đâu? Và vì sao một số điểm ảnh có $w_i$ đúng bằng 0?

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 3.6))
im = show_w(ax, w, "w sau 146 lần sửa")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()
touched = (Xtr[log["i"]] > 0).any(axis=0)
print(f"{(w > 0).sum()} trọng số dương, {(w < 0).sum()} âm, {(w == 0).sum()} bằng đúng 0")
print(
    f"điểm ảnh chưa từng có nét trong 146 ảnh gây lỗi: {(~touched).sum()} — trùng với số trọng số bằng 0: {np.all((w == 0) == ~touched)}"
)

Vùng cam là một dải dọc ở giữa ảnh, nơi nét số 1 đứng, cùng một mảng ở phía dưới. Vùng xanh nằm hai bên dải đó,
nơi số 7 có nét mà số 1 không có: thanh ngang và nét chéo. Hình này không phải một bức ảnh số 7 “trung bình”:
nó là tổng của 146 ảnh gây lỗi, cộng hoặc trừ, nên chỉ giữ những nét giúp *phân biệt* hai lớp.

Trọng số bằng 0 nằm chủ yếu ở viền. Mỗi lần sửa chỉ cộng $\pm\eta x$, nên điểm ảnh nào chưa từng có nét trong ảnh gây
lỗi nào thì trọng số của nó vẫn là 0 như lúc đầu.

### Một ảnh được quyết định thế nào
Tích từng phần $x_i w_i$ cho biết điểm ảnh nào góp phần nào vào $z$. Lấy số 7 và số 1 “điển hình” của tập kiểm tra
(ảnh có $z$ gần trung vị của lớp nó nhất; đây cũng là hai ảnh mà reel và demo dùng):

In [ ]:
z_te = Xte @ w + b


def typical(cls):
    idx = np.nonzero(yte == cls)[0]
    gap = np.round(np.abs(z_te[idx] - np.median(z_te[idx])), 9)  # số ảnh chẵn: hai ảnh giữa cách đều trung vị
    return idx[np.flatnonzero(gap == gap.min())[-1]]  # lấy ảnh sau, như reel và demo


fig, axes = plt.subplots(2, 3, figsize=(9, 6.2))
for row, j in enumerate([typical(1), typical(0)]):
    c = Xte[j] * w
    axes[row, 0].imshow(Xte[j].reshape(28, 28), cmap="gray_r")
    axes[row, 0].set_title(f"MNIST kiểm tra #{idx_te[j]}: x", fontsize=10)
    axes[row, 0].axis("off")
    show_w(axes[row, 1], w, "w")
    show_w(axes[row, 2], c, f"x ⊙ w: z = {vn(c.sum() + b, 2)}")
    print(
        f"#{idx_te[j]} (số {POS if yte[j] else NEG}): dương {vn(c[c > 0].sum(), 2)}, âm {vn(c[c < 0].sum(), 2)}, "
        f"b = {vn(b, 1)}  →  z = {vn(z_te[j], 2)}  →  “{POS if z_te[j] >= 0 else NEG}”"
    )
plt.tight_layout()
plt.show()

Chỉ những điểm có nét ($x_i > 0$) mới góp vào $z$; nền trắng nhân với bất kỳ $w_i$ nào cũng bằng 0.
Số 7 có nét nằm trên vùng xanh nên tổng dương thắng; số 1 nằm trên vùng cam nên tổng âm thắng.

## 7 · Mô hình sai ở đâu?
14 ảnh kiểm tra bị đoán sai, kèm $z$ của chúng. $z$ gần 0 nghĩa là “suýt đúng”.

In [ ]:
wrong = np.nonzero(pred != yte)[0]
fig, axes = plt.subplots(2, 7, figsize=(10, 3.4))
for ax, j in zip(axes.flat, wrong):
    ax.imshow(Xte[j].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"số {POS if yte[j] else NEG}\nz = {vn(z_te[j], 2)}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"{(yte[wrong] == 1).sum()} số 7 bị đoán là 1, {(yte[wrong] == 0).sum()} số 1 bị đoán là 7")

10 trong 14 lỗi là số 7, phần lớn có thanh ngang ngắn hoặc thân gần như thẳng đứng, nên nét rơi vào dải cam của số 1;
vài ảnh khác viết lệch hẳn sang một bên. Một nơ-ron chỉ có một “khuôn” $w$ duy nhất, nên những kiểu viết khác xa khuôn
đó sẽ bị đoán sai.

Phân bố $z$ của hai lớp cho thấy điều tương tự: hầu hết ảnh nằm xa ngưỡng 0, chỉ một ít nằm sai phía.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
bins = np.linspace(z_te.min(), z_te.max(), 60)
ax.hist(z_te[yte == 0], bins, color=ORANGE, alpha=0.75, label=f"ảnh số {NEG}")
ax.hist(z_te[yte == 1], bins, color=BLUE, alpha=0.75, label=f"ảnh số {POS}", hatch="//", edgecolor="white", lw=0)
ax.axvline(0, color="black", lw=1)
ax.set(xlabel="z", ylabel="số ảnh kiểm tra", title="z của từng ảnh kiểm tra; ngưỡng z = 0")
ax.legend(frameon=False)
plt.show()

## 8 · Thử thay đổi

Các ô dưới chạy lại cùng hàm `train()` với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### 🤔 Dự đoán trước: tốc độ học $\eta$
Nếu đổi $\eta = 0{,}1$ thành $0{,}01$ hoặc $1$, số lần sửa và độ chính xác thay đổi thế nào?

In [ ]:
for eta in [0.01, 0.1, 1.0]:
    w_e, b_e, log_e = train(Xtr, ytr, eta=eta, seed=0)
    acc_e = (step(Xte @ w_e + b_e) == yte).mean()
    print(
        f"η = {vn(eta, 2):>4}: {len(log_e['seen'])} lần sửa, đúng {pct(acc_e)}, "
        f"w/η giống hệt η = 0,1: {np.allclose(w_e / eta, w / 0.1)}"
    )

Không thay đổi gì. Vì $w$ và $b$ bắt đầu từ 0 và mỗi lần sửa đều cộng $\pm\eta x$, nên $w$ luôn bằng $\eta$ nhân một
tổng các ảnh; mà dấu của $z$ không đổi khi nhân $z$ với một số dương. Với perceptron xuất phát từ 0, $\eta$ chỉ đổi
độ lớn của $w$, không đổi quyết định nào. (Ở Bài 05, với gradient descent, $\eta$ sẽ quan trọng.)

### Thứ tự ảnh (seed)
Thứ tự duyệt ảnh thì có ảnh hưởng: mỗi lần sửa phụ thuộc vào $w$ hiện tại.

In [ ]:
for seed in range(5):
    w_s, b_s, log_s = train(Xtr, ytr, seed=seed)
    print(f"seed {seed}: {len(log_s['seen'])} lần sửa, đúng {pct((step(Xte @ w_s + b_s) == yte).mean())}")

Từ 144 đến 163 lần sửa, 98,94% đến 99,49%. Khi so hai mô hình, chênh lệch nhỏ hơn khoảng dao động giữa các seed này
thì chưa nói được bên nào tốt hơn.

### Học đến khi không còn sai
Nếu hai lớp **tách được bằng một đường thẳng** (ở đây là một siêu phẳng trong không gian 784 chiều),
định lý hội tụ của Novikoff (1962) bảo đảm quy tắc perceptron dừng sau hữu hạn lần sửa.
Thử lặp nhiều lượt đến khi một lượt không sai ảnh huấn luyện nào:

In [ ]:
t0 = time.perf_counter()
w_c, b_c, log_c = train(Xtr, ytr, epochs=200)
errs = log_c["epoch_errors"]
print(
    f"{len(errs)} lượt, {vn(len(log_c['seen']), 0)} lần sửa ({vn(time.perf_counter() - t0, 1)} giây); lỗi mỗi lượt: {errs[:6]} … {errs[-3:]}"
)
print(
    f"huấn luyện đúng {pct((step(Xtr @ w_c + b_c) == ytr).mean())}, kiểm tra đúng {pct((step(Xte @ w_c + b_c) == yte).mean())}"
)
assert errs[-1] == 0

Tập 7/1 tách được hoàn toàn: sau 93 lượt, mô hình đúng 100% ảnh huấn luyện. Nhưng trên ảnh mới chỉ tăng từ 99,35% lên
99,63%, và phải trả gấp 93 lần thời gian. Đúng hết tập huấn luyện không có nghĩa là đúng hết ảnh chưa thấy (Bài 08).

### Cặp chữ số khác
7 và 1 trông rất khác nhau. Đổi `pair` để thử cặp khó hơn, ví dụ 4 và 9 hoặc 3 và 8:

In [ ]:
for pair in [(7, 1), (0, 1), (3, 8), (4, 9), (5, 3)]:
    Xa, ya, _ = binary_set(train_img, train_lab, *pair)
    Xb, yb, _ = binary_set(test_img, test_lab, *pair)
    w_p, b_p, log_p = train(Xa, ya)
    print(
        f"{pair[0]} hay {pair[1]}: {vn(len(ya), 0)} ảnh, {len(log_p['seen'])} lần sửa, kiểm tra đúng {pct((step(Xb @ w_p + b_p) == yb).mean())}"
    )

Với các cặp giống nhau (4/9, 3/8, 5/3), một lượt chỉ còn 93–95% và phải sửa 5–6 lần nhiều hơn. Một khuôn tuyến tính
không đủ phân biệt hai chữ số có nhiều nét chung.

## 9 · Giới hạn: XOR

Mặt quyết định $z = 0$ của một nơ-ron là một đường thẳng (trong 2 chiều). Bài toán XOR có 4 điểm:
$(0,0)$ và $(1,1)$ nhãn 0, $(0,1)$ và $(1,0)$ nhãn 1. Không đường thẳng nào tách được chúng, nên quy tắc perceptron
sửa mãi mà không bao giờ đúng cả 4 (Minsky & Papert, 1969). Ta chạy đúng cách reel làm: xuất phát
$w = (0{,}3; -0{,}2)$, $b = 0{,}1$, mỗi bước sửa điểm sai đầu tiên tính từ sau điểm vừa sửa.

In [ ]:
P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
Y = np.array([0, 1, 1, 0])
w2, b2 = np.array([0.3, -0.2]), 0.1
history, cursor = [], 0
for _ in range(16):
    correct = step(P @ w2 + b2) == Y
    history.append((w2.copy(), b2, int(correct.sum())))
    if correct.all():
        break
    j = next((cursor + q) % 4 for q in range(4) if not correct[(cursor + q) % 4])
    d = Y[j] - int(P[j] @ w2 + b2 >= 0)
    w2, b2 = w2 + 0.1 * d * P[j], b2 + 0.1 * d
    cursor = (j + 1) % 4
history.append((w2.copy(), b2, int((step(P @ w2 + b2) == Y).sum())))
print("số điểm đúng sau mỗi lần sửa:", [h[2] for h in history])
assert max(h[2] for h in history) < 4

Không bao giờ quá 3/4. Hình dưới vẽ 4 điểm (tròn cam = nhãn 0, vuông xanh = nhãn 1), vài đường $z = 0$ trên đường đi
(xám) và đường cuối cùng (xanh lá; vùng tô xanh nhạt là nơi nơ-ron đoán 1). Đường nào cũng để ít nhất một điểm
nằm sai phía.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-0.5, 1.5, 201), np.linspace(-0.5, 1.5, 201))
grid = np.c_[g1.ravel(), g2.ravel()]
fig, ax = plt.subplots(figsize=(4.4, 4.4))
for wk, bk, _ in history[:-1:4]:  # vài đường trên đường đi (đường z = 0 có thể thẳng đứng, nên vẽ bằng contour)
    ax.contour(g1, g2, (grid @ wk + bk).reshape(g1.shape), levels=[0], colors=GREY, linewidths=1)
wl, bl, n_ok = history[-1]
zl = (grid @ wl + bl).reshape(g1.shape)
ax.contourf(g1, g2, zl, levels=[0, np.inf], colors=[BLUE], alpha=0.12)  # vùng đoán 1
ax.contour(g1, g2, zl, levels=[0], colors=GREEN, linewidths=2.5)
for p, lab in zip(P, Y):
    ax.scatter(*p, s=160, marker="s" if lab else "o", color=BLUE if lab else ORANGE, edgecolor="white", lw=2, zorder=3)
ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x₁", ylabel="x₂")
ax.set_title(f"XOR: đường cuối (xanh lá) đúng {n_ok}/4", fontsize=11)
ax.set_aspect("equal")
plt.show()

Lời giải là thêm một **lớp ẩn** với hàm kích hoạt phi tuyến: [Bài 03 · MLP và XOR](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mlp-va-xor/).

## 10 · Từ một nơ-ron lên một lớp: $z = xW + b$

Đặt $M$ nơ-ron cạnh nhau, mỗi nơ-ron một cột trọng số, ta được ma trận $W$ cỡ $784 \times M$.
Một ảnh (vectơ hàng $x$, cỡ $1 \times 784$) cho ra $M$ giá trị cùng lúc: $z = xW + b$, tốn $784 \cdot M$ MAC.
Xếp nhiều ảnh thành các hàng của $X$, cả tập được tính bằng một phép nhân ma trận $Z = XW + b$.

Nơ-ron của ta là trường hợp $M = 1$. Mạng MLP mà MLSysBook (và khóa học từ Bài 02) dùng cho MNIST là 784 → 128 → 64 → 10:

In [ ]:
layers = [784, 128, 64, 10]
total = 0
for n_in, n_out in itertools.pairwise(layers):
    total += n_in * n_out
    print(f"lớp {n_in:>3} → {n_out:>3}: W cỡ {n_in}×{n_out}, {vn(n_in * n_out, 0):>7} MAC")
print(f"tổng: {vn(total, 0)} MAC cho mỗi ảnh, so với 784 của một nơ-ron")
assert total == 109184

Một lớp 10 nơ-ron (một nơ-ron cho mỗi chữ số, chọn nơ-ron có $z$ lớn nhất) đã đọc được cả 10 chữ số.
Quy tắc perceptron mở rộng tự nhiên: khi đoán sai, cộng $\eta x$ vào cột của chữ số đúng và trừ $\eta x$ khỏi cột
của chữ số bị đoán nhầm. Một lượt qua 60.000 ảnh:

In [ ]:
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(int)
X_all_te, y_all_te = test_img.reshape(-1, 784) / 255.0, test_lab.astype(int)
W, b10, fixes = np.zeros((784, 10)), np.zeros(10), 0
for i in np.random.default_rng(0).permutation(len(y_all)):
    k = int(np.argmax(X_all[i] @ W + b10))
    if k != y_all[i]:
        W[:, y_all[i]] += 0.1 * X_all[i]
        b10[y_all[i]] += 0.1
        W[:, k] -= 0.1 * X_all[i]
        b10[k] -= 0.1
        fixes += 1
Z = X_all_te @ W + b10  # 10.000 ảnh × 10 nơ-ron, một phép nhân ma trận
print(
    f"Z có cỡ {Z.shape}; {vn(fixes, 0)} lần sửa; đúng {pct((Z.argmax(1) == y_all_te).mean(), 1)} trên 10.000 ảnh kiểm tra"
)

fig, axes = plt.subplots(1, 10, figsize=(10, 1.5))
for d, ax in enumerate(axes):
    show_w(ax, W[:, d], f"cột {d}")
plt.show()

88,7% trên 10 chữ số: tốt, nhưng mạng 784 → 128 → 64 → 10 của khóa học đạt 97,27% (demo Bài 04). Mỗi cột của $W$ vẫn chỉ
là một khuôn tuyến tính cho một chữ số (hình trên). Để vượt qua, mạng cần lớp ẩn, hàm kích hoạt trơn và cách học bằng
gradient: đó là nội dung của Chương 1–2.

## Tóm tắt
- Một perceptron tính $z = \sum_i x_i w_i + b$ (784 MAC cho một ảnh MNIST) rồi quyết định bằng hàm bậc thang.
- Quy tắc Rosenblatt chỉ sửa khi sai: $w \leftarrow w + \eta (y - \hat y) x$. Một lượt, 146 lần sửa: 47,53% → 99,35%.
- Trọng số đọc được như một bức ảnh: xanh là nét của “7”, cam là nét của “1”.
- Xuất phát từ 0, $\eta$ không đổi quyết định nào; thứ tự ảnh thì có (98,94–99,49% qua 5 seed).
- Dữ liệu tách được tuyến tính thì quy tắc hội tụ (Novikoff); XOR thì không bao giờ.
- Nhiều nơ-ron là một lớp, $z = xW + b$: phép nhân ma trận.

## Câu hỏi ôn tập
1. Vì sao với $w = 0$, $b = 0$, nơ-ron đoán “7” cho mọi ảnh?
2. Một điểm ảnh có $w_i$ dương lớn. Nét mực ở điểm đó đẩy quyết định về phía nào?
3. Sau khi huấn luyện, 297 trọng số vẫn bằng đúng 0. Vì sao?
4. Nhân $\eta$ lên 10 lần thì dự đoán có thay đổi không? Vì sao?
5. Một lớp $M = 10$ nơ-ron trên ảnh 784 điểm ảnh tốn bao nhiêu MAC mỗi ảnh?
6. Vì sao không một perceptron nào giải được XOR?

<details><summary>Gợi ý đáp án</summary>

1. $z = 0$ với mọi ảnh, và $f(0) = 1$ (“7”).
2. Về “7”: $x_i > 0$ nhân $w_i > 0$ làm $z$ tăng.
3. Mỗi lần sửa cộng $\pm\eta x$; những điểm ảnh không có nét trong bất kỳ ảnh nào trong 146 ảnh gây lỗi
   thì không bao giờ được cộng gì.
4. Không: $w$ và $b$ cùng nhân 10 lần nên $z$ nhân 10, dấu của $z$ giữ nguyên.
5. $784 \times 10 = 7.840$ MAC.
6. Mặt quyết định của một nơ-ron là một đường thẳng, mà không đường thẳng nào để $(0,1)$, $(1,0)$ một bên
   và $(0,0)$, $(1,1)$ bên kia.
</details>

## Tìm hiểu thêm
- MLSysBook, [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html): ví dụ MNIST và ký hiệu của bài này.
- F. Rosenblatt (1958), [The Perceptron](https://doi.org/10.1037/h0042519).
- A. Novikoff (1962), [On convergence proofs for perceptrons](https://web.engr.oregonstate.edu/~huanlian/teaching/ML/2018spring/extra/novikoff-1963.pdf).
- M. Minsky & S. Papert (1969), [Perceptrons](https://doi.org/10.7551/mitpress/11301.001.0001).
- Y. Freund & R. Schapire (1999), [Large margin classification using the perceptron algorithm](https://doi.org/10.1023/A:1007662407062):
  perceptron có bình chọn, cải tiến cho dữ liệu không tách được.

**Bài tiếp theo:** [Bài 02 · Hàm kích hoạt](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-kich-hoat/): vì sao thay hàm bậc thang bằng hàm trơn.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")